# 🔬 Comprehensive Hyperparameter Tuning: CoNLL-2003

**Goal**: Systematically explore hyperparameters for olfactory-NER

**8 Experiments**:
1. Baseline (ReLU) - Reference
2. Baseline (GELU) - Activation switch
3. More Receptors (256)
4. More Glomeruli (64)
5. Larger LSTM (512)
6. Lower Dropout (0.2)
7. Larger Batch (64)
8. Strong Regularization

**Automation**:
- ✅ Auto-save models to Google Drive
- ✅ Auto-save results to GitHub
- ✅ Generate comparison visualizations
- ✅ Track all metrics

## 1. Setup

In [ ]:
# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repository
!git clone https://github.com/bhushan1729/olfaction-inspired-ner.git
%cd olfaction-inspired-ner

In [ ]:
# Install dependencies
!pip install -q torch numpy scikit-learn seqeval matplotlib seaborn pandas tqdm tensorboard

In [ ]:
# Download GloVe
import os
if not os.path.exists('./data/glove.6B.300d.txt'):
    print("Downloading GloVe...")
    !mkdir -p data
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip -O data/glove.6B.zip
    !unzip -q data/glove.6B.zip -d data/
    !rm data/glove.6B.zip
    print("✓ Done")
else:
    print("✓ GloVe exists")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create model save directory
model_dir = '/content/drive/MyDrive/olfaction_ner/models'
!mkdir -p {model_dir}
print(f"✓ Models will be saved to: {model_dir}")

## 3. Helper Functions

In [ ]:
import json
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Create folders
!mkdir -p comparison_plots
!mkdir -p experiment_results/CoNLL-2003

# Track all results
all_experiments = []

def save_experiment_to_drive(exp_name, save_dir, model_dir='/content/drive/MyDrive/olfaction_ner/models'):
    """Save best model to Google Drive."""
    src = f"{save_dir}/best_model.pt"
    dst = f"{model_dir}/{exp_name}_best.pt"
    shutil.copy(src, dst)
    print(f"✓ Saved model to Drive: {dst}")

def save_experiment_results(exp_name, config, results, save_dir):
    """Save experiment results locally."""
    exp_dir = f"experiment_results/CoNLL-2003/{exp_name}"
    os.makedirs(exp_dir, exist_ok=True)
    
    # Save metadata
    metadata = {'experiment_name': exp_name, 'config': config}
    with open(f"{exp_dir}/metadata.json", 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Save results
    with open(f"{exp_dir}/results.json", 'w') as f:
        json.dump(results, f, indent=2)
    
    # Track globally
    all_experiments.append({
        'name': exp_name,
        'config': config,
        'results': results
    })
    
    print(f"✓ Saved results to: {exp_dir}")

def plot_training_curves(exp_name, results, save_path):
    """Plot training and validation curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = [e['epoch'] for e in results['epochs']]
    train_loss = [e['train']['total_loss'] for e in results['epochs']]
    valid_f1 = [e['valid']['f1'] for e in results['epochs']]
    
    # Loss curve
    axes[0].plot(epochs, train_loss, 'o-', label='Train Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{exp_name}: Training Loss')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    # F1 curve
    axes[1].plot(epochs, valid_f1, 'o-', label='Valid F1', color='green')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1 Score')
    axes[1].set_title(f'{exp_name}: Validation F1')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved curves to: {save_path}")

def plot_per_entity(exp_name, results, save_path):
    """Plot per-entity F1 scores."""
    per_entity = results['test']['per_entity']
    entities = [k for k in per_entity.keys() if k not in ['micro avg', 'macro avg', 'weighted avg']]
    scores = [per_entity[e] for e in entities]
    
    plt.figure(figsize=(10, 5))
    plt.bar(entities, scores, color='steelblue', alpha=0.8)
    plt.xlabel('Entity Type')
    plt.ylabel('F1 Score')
    plt.title(f'{exp_name}: Per-Entity F1 Scores')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1.0)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved per-entity plot to: {save_path}")

print("✓ Helper functions loaded")

---
# Experiments
---

## Experiment 1: Baseline (ReLU)

In [ ]:
# Train
!python src/train.py \
  --config config/experiments.yaml \
  --experiment olfactory_full \
  --save_dir results/tuning/exp1_baseline_relu

In [ ]:
# Save and visualize
with open('results/tuning/exp1_baseline_relu/results.json') as f:
    results = json.load(f)

config = {'activation': 'relu', 'num_receptors': 128, 'num_glomeruli': 32, 'lstm_hidden': 256, 'dropout': 0.5, 'batch_size': 32, 'lambda_sparse': 0.001, 'lambda_diverse': 0.01}

save_experiment_to_drive('exp1_baseline_relu', 'results/tuning/exp1_baseline_relu')
save_experiment_results('exp1_baseline_relu', config, results, 'results/tuning/exp1_baseline_relu')
plot_training_curves('Exp1: Baseline (ReLU)', results, 'comparison_plots/exp1_curves.png')
plot_per_entity('Exp1: Baseline (ReLU)', results, 'comparison_plots/exp1_entities.png')

print(f"\n✅ Experiment 1 Complete: Test F1 = {results['test']['f1']:.4f}")

## Experiment 2: Baseline (GELU)

In [ ]:
!python src/train.py \
  --config config/tuning_experiments.yaml \
  --experiment activation_gelu \
  --save_dir results/tuning/exp2_baseline_gelu

In [ ]:
with open('results/tuning/exp2_baseline_gelu/results.json') as f:
    results = json.load(f)

config = {'activation': 'gelu', 'num_receptors': 128, 'num_glomeruli': 32, 'lstm_hidden': 256, 'dropout': 0.5, 'batch_size': 32, 'lambda_sparse': 0.001, 'lambda_diverse': 0.01}

save_experiment_to_drive('exp2_baseline_gelu', 'results/tuning/exp2_baseline_gelu')
save_experiment_results('exp2_baseline_gelu', config, results, 'results/tuning/exp2_baseline_gelu')
plot_training_curves('Exp2: Baseline (GELU)', results, 'comparison_plots/exp2_curves.png')
plot_per_entity('Exp2: Baseline (GELU)', results, 'comparison_plots/exp2_entities.png')

print(f"\n✅ Experiment 2 Complete: Test F1 = {results['test']['f1']:.4f}")

## Experiment 3: More Receptors (256)

In [ ]:
!python src/train.py \
  --config config/tuning_experiments.yaml \
  --experiment more_receptors \
  --save_dir results/tuning/exp3_more_receptors

In [ ]:
with open('results/tuning/exp3_more_receptors/results.json') as f:
    results = json.load(f)

config = {'activation': 'gelu', 'num_receptors': 256, 'num_glomeruli': 64, 'lstm_hidden': 256, 'dropout': 0.5, 'batch_size': 32, 'lambda_sparse': 0.001, 'lambda_diverse': 0.01}

save_experiment_to_drive('exp3_more_receptors', 'results/tuning/exp3_more_receptors')
save_experiment_results('exp3_more_receptors', config, results, 'results/tuning/exp3_more_receptors')
plot_training_curves('Exp3: More Receptors (256)', results, 'comparison_plots/exp3_curves.png')
plot_per_entity('Exp3: More Receptors', results, 'comparison_plots/exp3_entities.png')

print(f"\n✅ Experiment 3 Complete: Test F1 = {results['test']['f1']:.4f}")

## Experiment 4-8: Continue Pattern...

*Similar cells for remaining experiments*

---
# Final Comparison
---

## Comparison Table

In [ ]:
import pandas as pd

# Build comparison table
comparison_data = []
for exp in all_experiments:
    row = {
        'Experiment': exp['name'],
        'Activation': exp['config'].get('activation', 'N/A'),
        'Receptors': exp['config'].get('num_receptors', 'N/A'),
        'Glomeruli': exp['config'].get('num_glomeruli', 'N/A'),
        'LSTM': exp['config'].get('lstm_hidden', 'N/A'),
        'Dropout': exp['config'].get('dropout', 'N/A'),
        'Batch': exp['config'].get('batch_size', 'N/A'),
        'λ_sparse': exp['config'].get('lambda_sparse', 'N/A'),
        'λ_diverse': exp['config'].get('lambda_diverse', 'N/A'),
        'Test F1': f"{exp['results']['test']['f1']:.4f}",
        'Precision': f"{exp['results']['test']['precision']:.4f}",
        'Recall': f"{exp['results']['test']['recall']:.4f}"
    }
    comparison_data.append(row)

df = pd.DataFrame(comparison_data)
df = df.sort_values('Test F1', ascending=False)

print("\n" + "="*80)
print(" "*25 + "EXPERIMENT COMPARISON")
print("="*80)
print(df.to_string(index=False))
print("="*80)

# Save table
df.to_csv('comparison_plots/comparison_table.csv', index=False)
print("\n✓ Saved comparison table")

## Push to GitHub

In [ ]:
# Setup git
!git config user.name "Colab Experiment"
!git config user.email "colab@experiment.local"

# Add all results
!git add experiment_results/
!git add comparison_plots/

# Commit
!git commit -m "Add hyperparameter tuning results - 8 experiments"

# Push (for public repo)
!git push origin main

print("\n✅ Results pushed to GitHub!")
print("View at: https://github.com/bhushan1729/olfaction-inspired-ner/tree/main/experiment_results")